In [5]:
!pip3 install boto3

In [8]:
import boto3
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, year, month, dayofmonth, dayofweek, lower, regexp_replace, trim

# =========================
# CONFIG S3
# =========================

s3 = boto3.client("s3")

bucket_raw = "last-mile-optimization-raw-gabriel"
bucket_trusted = "last-mile-optimization-trusted-gabriel"

input_key = "feriados_brasil_2017.csv"

local_input = "/tmp/feriados.csv"
local_output_full = "/tmp/output_full"
local_output_dates = "/tmp/output_dates"

# =========================
# DOWNLOAD DO S3
# =========================

s3.download_file(bucket_raw, input_key, local_input)

# =========================
# SPARK
# =========================

spark = SparkSession.builder \
    .appName("etl_holidays_br_2017_csv") \
    .getOrCreate()

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(local_input)

# =========================
# TRATAMENTO
# =========================

df = df.withColumn("date", col("date").cast("date"))

df = df.withColumn("year", year(col("date"))) \
       .withColumn("month", month(col("date"))) \
       .withColumn("day", dayofmonth(col("date"))) \
       .withColumn("day_number", dayofweek(col("date")))

df = df.withColumn(
    "day_of_week",
    when(col("day_number") == 1, "sunday")
    .when(col("day_number") == 2, "monday")
    .when(col("day_number") == 3, "tuesday")
    .when(col("day_number") == 4, "wednesday")
    .when(col("day_number") == 5, "thursday")
    .when(col("day_number") == 6, "friday")
    .when(col("day_number") == 7, "saturday")
)

df = df.withColumn(
    "weekend",
    when(col("day_number").isin(1, 7), "yes").otherwise("no")
)

df = df.withColumn("holiday", lit("yes"))

df = df.withColumn(
    "national",
    when(col("global") == True, "yes").otherwise("no")
)

df = df.withColumn(
    "regions",
    when(col("counties").isNull(), "all").otherwise(col("counties"))
)

df = df.withColumn("nome_local", col("localName"))
df = df.withColumn("english_name", col("name"))
df = df.withColumn("country_code", col("countryCode"))
df = df.withColumn("types", col("types"))

colunas_texto = [
    "day_of_week","weekend","holiday","nome_local","english_name",
    "country_code","national","regions","types"
]

for coluna in colunas_texto:
    df = df.withColumn(coluna, lower(col(coluna)))
    df = df.withColumn(coluna, regexp_replace(col(coluna), "[\\s\\-]+", "_"))
    df = df.withColumn(coluna, regexp_replace(col(coluna), "[^a-z0-9_áàâãéèêíïóôõöúç]", ""))
    df = df.withColumn(coluna, regexp_replace(col(coluna), "_+", "_"))
    df = df.withColumn(coluna, regexp_replace(col(coluna), "^_|_$", ""))
    df = df.withColumn(coluna, trim(col(coluna)))
    df = df.withColumn(
        coluna,
        when(col(coluna).isNull() | (trim(col(coluna)) == ""), "null").otherwise(col(coluna))
    )

df_final = df.select(
    "date","year","month","day","day_of_week","weekend","holiday",
    "nome_local","english_name","country_code","national","regions","types"
)

df_datas = df_final.select("date")

# =========================
# SALVAR LOCAL
# =========================

df_final.coalesce(1).write.mode("overwrite").option("header", "true").csv(local_output_full)
df_datas.coalesce(1).write.mode("overwrite").option("header", "true").csv(local_output_dates)

# =========================
# UPLOAD PARA S3
# =========================

def upload_folder(local_path, bucket, s3_path):
    for file in os.listdir(local_path):
        if file.startswith("part-"):
            s3.upload_file(
                os.path.join(local_path, file),
                bucket,
                f"{s3_path}/{file}"
            )

upload_folder(local_output_full, bucket_trusted, "holidays/full")
upload_folder(local_output_dates, bucket_trusted, "holidays/dates")

print("ETL completed successfully!")
print("Data uploaded to S3 (trusted)")

etl completed successfully!
file saved: etl_holidays_brazil_2026.csv
file saved: etl_holidays_brazil_2026_dates.csv
